# 案例 06 · 日志写满云盘，笔记服务写入失败

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

日志按天切割但从不清理，大促期间日志量翻倍写满 /data；同时有一个已删除但被进程持有的日志文件，`rm` 后空间不释放，服务写入报 `Errno 28`。

## 练习目标

- 用 `df -h` / `du -sh` 定位空间占用
- 用 `lsof +L1` 找出 deleted-open 文件
- 清理 + 重启释放句柄，恢复写入
- 用 `disk_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-06-disk-full")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_disk_full()
used, _ = lab.disk_usage()
print(f"已注入：/data 使用率 {used / lab.quota_mb * 100:.0f}%")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**分区使用率**

In [ ]:
print(lab.sh("df -h /data"))

**目录占用**

In [ ]:
print(lab.sh("du -sh /data/*"))

**deleted-open 文件**

In [ ]:
print(lab.sh("lsof +L1"))

**应用日志**

In [ ]:
print(lab.sh("tail -n 3 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
used, _ = lab.disk_usage()
fault = lab.state["faults"]["disk"]
print(f"/data 使用率: {used / lab.quota_mb * 100:.0f}%")
print("deleted-open:", fault["deleted_open"])
print("判断：空间耗尽且存在 deleted 句柄 → 需要“清理 + 释放句柄”，只 rm 无效")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
# 1) 按保留策略清理历史日志（保留最近 2 个）
logs = sorted((lab.data / "logs").glob("app-*.log"), key=lambda f: f.stat().st_mtime)
for f in logs[:-2]:
    f.unlink(missing_ok=True)

# 2) 重启持有 deleted 文件句柄的进程，释放空间
svc = lab.state["services"]["xhs-note-api"]
lab.kill_service()
lab.start_api(port=svc["port"])
lab.state["faults"].pop("disk", None)
lab._save()
time.sleep(0.5)
used, _ = lab.disk_usage()
print(f"处置后使用率: {used / lab.quota_mb * 100:.0f}%")


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('使用率 < 70%', bool(lab.disk_usage()[0] / lab.quota_mb < 0.7))
lab.check('服务健康', bool('HTTP_CODE=200' in lab.curl(lab.api + '/healthz')))

## 步骤 6 · 自动化沉淀

脚本用 `df/du/lsof +L1` 取证，按保留策略清理并压缩日志，重启持有 deleted 句柄的进程，最后校验水位回落。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_disk_full()
used, _ = lab.disk_usage()
out = lab.auto('disk_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. `df` 100% 但 `du` 只有 60%，可能有哪些原因？
2. 生产重启释放句柄前，如何把影响面压到最小？
3. 日志要保留 180 天，本地盘方案该如何调整？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')